<a href="https://colab.research.google.com/github/dkang1630/Conductor_Image_Classification/blob/main/U_net_Prediction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install git+https://github.com/facebookresearch/segment-anything.git
!pip install matplotlib opencv-python pycocotools

!wget https://dl.fbaipublicfiles.com/segment_anything/sam_vit_h_4b8939.pth

# Copy to Google Drive
!cp sam_vit_h_4b8939.pth "/content/drive/My Drive/Conductor_Image_Detection/Images/Lab_experiment/U-Net/SAM/Files"

  Cloning https://github.com/facebookresearch/segment-anything.git to /tmp/pip-req-build-j5odizgx
  Running command git clone --filter=blob:none --quiet https://github.com/facebookresearch/segment-anything.git /tmp/pip-req-build-j5odizgx
  Resolved https://github.com/facebookresearch/segment-anything.git to commit dca509fe793f601edb92606367a655c15ac00fdf
  Preparing metadata (setup.py) ... done
--2025-01-29 21:22:14--  https://dl.fbaipublicfiles.com/segment_anything/sam_vit_h_4b8939.pth
Resolving dl.fbaipublicfiles.com (dl.fbaipublicfiles.com)... 13.226.210.78, 13.226.210.25, 13.226.210.111, ...
Connecting to dl.fbaipublicfiles.com (dl.fbaipublicfiles.com)|13.226.210.78|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 2564550879 (2.4G) [binary/octet-stream]
Saving to: ‘sam_vit_h_4b8939.pth.1’

sam_vit_h_4b8939.pt 100%[===================>]   2.39G   103MB/s    in 24s     

2025-01-29 21:22:38 (102 MB/s) - ‘sam_vit_h_4b8939.pth.1’ saved [2564550879/2564550879]



In [ ]:
from tensorflow.keras.models import load_model
from google.colab import drive
import os
import numpy as np
import random
import cv2
from tensorflow.keras.layers import Conv2D, BatchNormalization, Activation, MaxPool2D, Conv2DTranspose, Concatenate, Input
from tensorflow.keras.models import Model
from tensorflow.keras.callbacks import ModelCheckpoint, ReduceLROnPlateau, EarlyStopping, CSVLogger
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from segment_anything import SamPredictor, sam_model_registry

# Mount Google Drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
#Set random seed
SEED = 42
os.environ["PYTHONHASHSEED"] = str(SEED)
np.random.seed(SEED)
random.seed(SEED)

print("Numpy random array:", np.random.rand(3))  # Generate a random array with 3 elements
print("Python random number:", random.random())  # Generate a random float

Numpy random array: [0.37454012 0.95071431 0.73199394]
Python random number: 0.6394267984578837


In [ ]:
# Define base paths
base_path = "/content/drive/My Drive/Conductor_Image_Detection/Images/Lab_experiment/U-Net"
original_base_path = os.path.join(base_path, "Original")
binary_mask_base_path = os.path.join(base_path, "Binary_Masked")

original_train_base_path = os.path.join(original_base_path, "train")
original_test_base_path = os.path.join(original_base_path, "test")
original_validation_base_path = os.path.join(original_base_path, "validation")

binary_mask_train_base_path = os.path.join(binary_mask_base_path, "train")
binary_mask_test_base_path = os.path.join(binary_mask_base_path, "test")
binary_mask_validation_base_path = os.path.join(binary_mask_base_path, "validation")

files_dir = os.path.join(base_path, "Files")
Unet_model_file = os.path.join(files_dir, "Unet_model.keras")
csv_file = os.path.join(files_dir, "Unet_log.csv")

#SAM model path
drive_path = "/content/drive/My Drive/Conductor_Image_Detection/Images/Lab_experiment/U-Net/SAM/Files/sam_vit_h_4b8939.pth"
local_path = "sam_vit_h_4b8939.pth"

# Check if model exists in Drive
import os
if os.path.exists(drive_path):
    !cp "{drive_path}" .
    print("Model loaded from Drive")
else:
    !wget https://dl.fbaipublicfiles.com/segment_anything/sam_vit_h_4b8939.pth
    !cp sam_vit_h_4b8939.pth "{drive_path}"
    print("Downloaded new model and saved to Drive")

# Initialize SAM
from segment_anything import sam_model_registry, SamPredictor
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"
sam = sam_model_registry["vit_h"](checkpoint=local_path)
sam.to(device=device)
predictor = SamPredictor(sam)
print("SAM initialized successfully")

✓ Model loaded from Drive
✓ SAM initialized successfully


In [ ]:
height = 128
width = 128

In [ ]:
#Conv Block
def conv_block(inputs, num_filters):
    x = Conv2D(num_filters, 3, padding="same")(inputs)
    x = BatchNormalization()(x)
    x = Activation("relu")(x)

    x = Conv2D(num_filters, 3, padding="same")(x)
    x = BatchNormalization()(x)
    x = Activation("relu")(x)

    return x

#Encoder Block
def encoder_block(inputs, num_filters):
    x = conv_block(inputs, num_filters)
    p = MaxPool2D((2, 2))(x)
    return x, p

#Decoder Block
def decoder_block(inputs, skip_features, num_filters):
    x = Conv2DTranspose(num_filters, (2, 2), strides=2, padding="same")(inputs)
    x = Concatenate() ([x, skip_features])
    x = conv_block(x, num_filters)
    return x

In [ ]:
def build_unet_model(input_shape):
    inputs = Input(input_shape)

    #Encoder
    s1, p1 = encoder_block(inputs, 64)
    s2, p2 = encoder_block(p1, 128)
    s3, p3 = encoder_block(p2, 256)
    s4, p4 = encoder_block(p3, 512)

    #Bridge
    b1 = conv_block(p4, 1024)

    #Decoder
    d1 = decoder_block(b1, s4, 512)
    d2 = decoder_block(d1, s3, 256)
    d3 = decoder_block(d2, s2, 128)
    d4 = decoder_block(d3, s1, 64)

    #Output
    outputs = Conv2D(1, 1, padding="same", activation="sigmoid")(d4)

    model = Model(inputs, outputs, name="U-Net")
    return model

In [ ]:
input_shape = (height, width, 3)
model = build_unet_model(input_shape)
model.load_weights(Unet_model_file)

In [ ]:
#Function to create a buffer zone around the mask
def create_expanded_mask(mask, buffer_pixels=10):
    # Ensure mask is binary (0 or 255)
    _, binary_mask = cv2.threshold(mask.astype(np.uint8), 127, 255, cv2.THRESH_BINARY)

    # Create horizontal-oriented kernel (wider horizontal expansion)
    kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (2*buffer_pixels + 1, 3))

    # Single dilation with large kernel
    expanded_mask = cv2.dilate(binary_mask, kernel, iterations=1)

    return expanded_mask

def preprocess_image(image_path, target_shape):
    image = cv2.imread(image_path)
    image = cv2.resize(image, (target_shape[1], target_shape[0]))
    image = image / 255.0  # Normalize pixel values
    image = np.expand_dims(image, axis=0)  # Add batch dimension
    return image

def apply_mask_to_image(original_image, mask, gray_value=100):
    # Threshold mask first (0-255 -> 0 or 255)
    _, binary_mask = cv2.threshold(mask.astype(np.uint8), 127, 255, cv2.THRESH_BINARY)

    # Convert to 3 channels using THRESHOLDED MASK
    mask_3d = np.stack([binary_mask] * 3, axis=-1)  # Use binary_mask not mask

    # Create gray background
    gray_background = np.full_like(original_image, gray_value, dtype=np.uint8)

    # Apply using 255 (not 1) for foreground
    masked_image = np.where(mask_3d == 255, original_image, gray_background)  # Changed from 1 to 255

    return masked_image

#assign points for SAM automatically based on mask
def get_auto_points(mask, image_shape):
    contours, _ = cv2.findContours(mask, cv2. RETR_EXTERNAL, cv2. CHAIN_APPROX_SIMPLE)
    if not contours:
        return None, None

    #Select the largest contour by area
    largest_contour = max(contours, key=cv2.contourArea)
    x, y, w, h = cv2.boudingRect(largest_contour)

    #Calculate mask edges
    mask_col_sum = np.sum(mask, axis=0)
    left_edge = np.argmax(mask_col_sum > 0)
    right_edge = len(mask_col_sum) - np.argmax(mask_col_sum[::-1] > 0)

    #input points (50px inside edges)
    input_points = [
        [left_edge + 50, y + h//2], #left side
        [right_edge - 50, y + h//2], #right side
        [x + w//2, y + 50]  #top center
    ]

    #background points
    background_points = [
        [image_shape[-1]-10, 10], #top right
        [10, image_shape[0]- 10], #bottom-left corner
        [image_shape[1]-10, image_shape[0]-10] #bottom right
    ]

    return (
        np.array(input_points, dtype=np.int32),
        np.array(background_points, dtype=np.int32)
    )


In [ ]:
test_image_dir = original_test_base_path  # Directory containing test images
predicted_mask_dir = binary_mask_test_base_path

sam_checkpoint = "sam_vit_h_4b8939.pth"
model_type = "vit_h"
device = "cuda" if torch.cuda.is_available() else "cpu"
# Initialize SAM
sam = sam_model_registry[model_type](checkpoint=sam_checkpoint)
sam.to(device=device)
predictor = SamPredictor(sam)

# Loop through each test image
for file_name in os.listdir(original_test_base_path):
    # Full path to the test image
    image_path = os.path.join(original_test_base_path, file_name)

    # Load the original image to get its dimensions
    original_image = cv2.imread(image_path)
    original_height, original_width = original_image.shape[:2]

    # Preprocess the image for prediction
    image = preprocess_image(image_path, input_shape)

    # Predict the mask using the model
    predicted_mask = model.predict(image)

    # Apply thresholding (for binary segmentation)
    predicted_mask = (predicted_mask > 0.5).astype(np.uint8)

    # Resize the predicted mask back to the original size
    resized_mask = cv2.resize(predicted_mask[0, :, :, 0], (original_width, original_height), interpolation=cv2.INTER_NEAREST)

    # Convert to 0-255 scale (critical for dilation)
    resized_mask = (resized_mask * 255).astype(np.uint8)  # 0 becomes 0, 1 becomes 255

    #create buffer zone around the resized mask
    buffer_pixels = 30
    expanded = create_expanded_mask(resized_mask, buffer_pixels)

    # SAM refinment
    try:
        input_points, background_points = get_auto_points(expanded, resized_mask.shape)
        if input_points is not None:
            predictor.set_image(original_image)

            #combine foreground and background points
            all_points = np.concatenate([input_points, background_points])
            all_labels = np.concatenate(
                [np.ones(input_points.shape[0]),  #foreground labels
                 np.zeros(background_points.shape[0] #background labels
                )]
            )

            #SAM prediction
            masks, scores, _ = predictor.predict(
                point_coords=all_points,
                point_labels=all_labels,
                multimask_output=False
            )

            #Get best SAM mask
            sam_mask = mask[0].astype(np.unit8) * 255

            #Use SAM mask if valid
            if no.any(sam_mask)
    # Apply the mask to the original image
    masked_image = apply_mask_to_image(original_image, expanded)

    # Save the masked image
    mask_save_path = os.path.join(predicted_mask_dir, f"masked_{file_name}")
    cv2.imwrite(mask_save_path, masked_image)

    print(f"Processed and saved masked image for {file_name}")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 472ms/step
Processed and saved masked image for img_31.jpg
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 894ms/step
Processed and saved masked image for img_34.jpg
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 829ms/step
Processed and saved masked image for img_97.jpg
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 853ms/step
Processed and saved masked image for img_157.jpg
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 932ms/step
Processed and saved masked image for img_2.jpg
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 475ms/step
Processed and saved masked image for img_5.jpg
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 486ms/step
Processed and saved masked image for img_14.jpg
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 521ms/step
Processed and saved masked image for img_17.jpg
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 497ms/step
Processed and saved masked image for img_20.jpg
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 482ms/step
Processed and saved masked image for img_23.jpg
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 447ms/step
Processed and saved masked image for img_26.jpg
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 469ms/step
Proce